# Gradient Descent — From Scratch (Batch, Mini-Batch, SGD)

We implement all three GD variants and compare their convergence behaviour
on a linear regression problem.

**Equations:**

Gradient of MSE cost w.r.t. $\boldsymbol{\theta}$:
$$\nabla_\theta J = \frac{1}{|B|} X_B^T (X_B \boldsymbol{\theta} - \mathbf{y}_B)$$

where $B$ is the batch (all samples for Batch GD, one for SGD, $b$ for Mini-Batch).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '../..')
from src.utils import set_style, plot_cost_history, regression_report

set_style()
np.random.seed(42)

## 1. Dataset

In [ ]:
# y = 1 + 2x₁ - 0.5x₂ + noise
m, n_feat = 300, 2
X_raw = np.random.randn(m, n_feat)
y     = 1 + 2 * X_raw[:, 0] - 0.5 * X_raw[:, 1] + np.random.randn(m) * 0.8

# Feature-scale + add bias
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_raw)
X = np.hstack([np.ones((m, 1)), X_scaled])   # (m, 3): [1, x1_scaled, x2_scaled]

print(f'X shape: {X.shape}  |  y shape: {y.shape}')

## 2. Helper: Cost Function

In [ ]:
def mse_cost(X, y, theta):
    r = X @ theta - y
    return float(r @ r) / (2 * len(y))


def gradient(X_b, y_b, theta):
    """Gradient of MSE for a batch (X_b, y_b)."""
    r = X_b @ theta - y_b
    return X_b.T @ r / len(y_b)

## 3. Batch Gradient Descent

In [ ]:
def batch_gd(X, y, alpha=0.05, n_iter=500):
    theta = np.zeros(X.shape[1])
    cost_hist = []
    for _ in range(n_iter):
        theta -= alpha * gradient(X, y, theta)   # use all m samples
        cost_hist.append(mse_cost(X, y, theta))
    return theta, cost_hist

theta_bgd, hist_bgd = batch_gd(X, y, alpha=0.05, n_iter=300)
print('Batch GD  θ:', theta_bgd)

## 4. Mini-Batch Gradient Descent

In [ ]:
def minibatch_gd(X, y, alpha=0.05, n_epochs=30, batch_size=32):
    m = len(y)
    theta     = np.zeros(X.shape[1])
    cost_hist = []

    for epoch in range(n_epochs):
        # Shuffle each epoch to avoid update patterns
        idx = np.random.permutation(m)
        X_shuf, y_shuf = X[idx], y[idx]

        for start in range(0, m, batch_size):
            X_b = X_shuf[start : start + batch_size]
            y_b = y_shuf[start : start + batch_size]
            theta -= alpha * gradient(X_b, y_b, theta)

        cost_hist.append(mse_cost(X, y, theta))  # record cost once per epoch

    return theta, cost_hist

theta_mgd, hist_mgd = minibatch_gd(X, y, alpha=0.05, n_epochs=300, batch_size=32)
print('Mini-Batch GD θ:', theta_mgd)

## 5. Stochastic Gradient Descent (SGD)

In [ ]:
def sgd(X, y, alpha=0.01, n_epochs=30):
    """SGD with learning rate decay (schedule): α_t = α0 / (1 + decay*t)."""
    m      = len(y)
    theta  = np.zeros(X.shape[1])
    cost_hist = []
    decay  = 0.001
    t      = 0

    for epoch in range(n_epochs):
        idx = np.random.permutation(m)
        for i in idx:
            X_i = X[[i]]
            y_i = y[[i]]
            a_t = alpha / (1 + decay * t)     # learning-rate schedule
            theta -= a_t * gradient(X_i, y_i, theta)
            t    += 1
        cost_hist.append(mse_cost(X, y, theta))

    return theta, cost_hist

theta_sgd, hist_sgd = sgd(X, y, alpha=0.05, n_epochs=300)
print('SGD θ:', theta_sgd)

## 6. Compare Convergence

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(hist_bgd, color='#2563EB', lw=2,   label='Batch GD')
ax.plot(hist_mgd, color='#16A34A', lw=1.5, label='Mini-Batch GD (b=32)')
ax.plot(hist_sgd, color='#DC2626', lw=1,   alpha=0.85, label='SGD')
ax.set_xlabel('Epoch')
ax.set_ylabel('Cost J(θ)')
ax.set_title('Batch vs Mini-Batch vs SGD — Convergence')
ax.legend()
plt.tight_layout()
plt.show()

print(f'Final cost — Batch: {hist_bgd[-1]:.4f} | Mini-batch: {hist_mgd[-1]:.4f} | SGD: {hist_sgd[-1]:.4f}')

## 7. Compare to Analytic (Normal Equation)

In [ ]:
theta_ols, *_ = np.linalg.lstsq(X, y, rcond=None)

print('Normal equation θ:  ', np.round(theta_ols, 4))
print('Batch GD          θ:', np.round(theta_bgd,  4))
print('Mini-Batch GD     θ:', np.round(theta_mgd,  4))
print('SGD               θ:', np.round(theta_sgd,  4))